Reads data from NPC simulations and saves trajectories for each NTR.

In [2]:
# import dependencies
import RMF
import pickle
import numpy as np
import os

In [3]:
def _has_depth_with_site(root, i):
    """ returns true if node subtree thru first child is at least i
        levels, including the root node itself, and the lead is a site """
#  print root, i, len(root.get_children())
    if (i==1) and root.get_name()=="site":
        return True
    c = root.get_children()
    if len(c) == 0:
        return False
    return _has_depth_with_site(c[0], i-1)

def _add_nodes(node, tf, type_prefixes, depth=0):
    '''
    node - rmf node to scan
    tf - typed factory
    type_prefixes - list of full type prefixes (e.g. "Nup1" for "Nup1N")

    adds only nodes whose type name begins with any of the specified type prefixes
    '''
    children = node.get_children()
    ret = []
    #print "inspecting", node.get_name()
    if len(children)==0:
        return ret
    if _has_depth_with_site(node, 3) and tf.get_is(children[0]):
        child_type = tf.get(children[0]).get_type_name()
        if any([child_type.startswith(tp) for tp in type_prefixes]):
            ret.append(children)
    for c in children:
        ret += _add_nodes(c, tf,  type_prefixes, depth+1)
    return ret

def get_trajectories_shape(kap_amount, start_t, end_t, step_t, frames_per_file):
    return [kap_amount, 3, int(((end_t - start_t) / step_t) * frames_per_file)]

def load_data(input_rmf_path, kap_radius, kap_amount, start_t, end_t, step_t, frames_per_file=104, one_frame_from_each=False):
    trajectories = np.zeros(shape=get_trajectories_shape(kap_amount, start_t, end_t, step_t, frames_per_file))

    for rmf_t in range(start_t, end_t, step_t):
        in_fh = RMF.open_rmf_file_read_only(f"{input_rmf_path}/{rmf_t}.movie.rmf")
        rff = RMF.ReferenceFrameFactory(in_fh)
        tf = RMF.TypedFactory(in_fh)
        # fg_types = [f"fg{x}" for x in range(32)]
        kap_types = [f"kap{kap_radius}"]

        # load data
        type2chains={}
        for i, kap_type in enumerate(kap_types):
            type2chains[kap_type] = _add_nodes(in_fh.get_root_node(), tf, [kap_type])
            
        # set frame
        for f_id, f in enumerate(in_fh.get_frames()):
            in_fh.set_current_frame(f)
    
            traj_i = int(f_id + ((rmf_t - start_t) / step_t) * frames_per_file)
            # read data
            for kap_i in range(kap_amount):
                coord = rff.get(type2chains["kap35"][0][kap_i]).get_translation()
                
                trajectories[kap_i, 0, traj_i] = coord[0] / 10
                trajectories[kap_i, 1, traj_i] = coord[1] / 10
                trajectories[kap_i, 2, traj_i] = coord[2] / 10
            if one_frame_from_each:
                break
    return trajectories

def load_and_save_data(in_path, out_path):
    x_coords, y_coords, z_coords = load_data(in_path, 0)
    with open(out_path, "wb") as f:
        pickle.dump([x_coords, y_coords, z_coords], f)
    

In [27]:
trajectories = load_data(
          input_rmf_path="/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/2/",
          kap_radius=35,
          kap_amount=250,
          start_t=40000,
          end_t=80000,
          step_t=100,
          frames_per_file=1,
          one_frame_from_each=True)
with open("spatial_markov_chain_data.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [ ]:
kap_amount = 250
start_t = 140000
end_t = 150000
step_t = 100
frames_per_file = 1
sims_range = range(1, 51)

bad_sims = []
for i in sims_range:
    # check that trajectory reached end_t time
    if not os.path.isfile(f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/divergences/{i}/{end_t}.movie.rmf"):
        print(f"sim {i} not long enough")
        bad_sims.append(i)
        continue

good_sims = [i for i in sims_range if i not in bad_sims]
# diverged_trajs = np.zeros(shape=[len(good_sims)] + get_trajectories_shape(kap_amount, start_t, end_t, step_t, frames_per_file))
for i in good_sims:
    print(f"loading sim {i}")
    trajectories = load_data(
          input_rmf_path=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/divergences/{i}",
          kap_radius=35,
          kap_amount=kap_amount,
          start_t=start_t,
          end_t=end_t,
          step_t=step_t,
          frames_per_file=frames_per_file,
          one_frame_from_each=True)
    with open(f"data/singles/{i}/140-150.pickle", "wb") as f:
        pickle.dump(trajectories, f)

loading sim 1
loading sim 2
loading sim 3
loading sim 4
loading sim 5
loading sim 6
loading sim 7
loading sim 8
loading sim 9
loading sim 10
loading sim 11
loading sim 12
loading sim 13
loading sim 14
loading sim 15
loading sim 16
loading sim 17
loading sim 18
loading sim 19


In [5]:
# Create merged pickle

arrays = []
for i in range(1, 51):
    with open(f"data/singles/{i}/110-130.pickle", "rb") as f:
        arrays.append(pickle.load(f))
trajectories = np.concatenate(arrays, axis=0)
with open(f"data/merged/110-130.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [9]:
arrays = []
for i in [j for j in range(1, 51)]:
    temp_arr = []
    with open(f"data/singles/{i}/110-130.pickle", "rb") as f:
        temp_arr.append(pickle.load(f))
    with open(f"data/singles/{i}/130-140.pickle", "rb") as f:
        temp_arr.append(pickle.load(f))
    arrays.append(np.concatenate(temp_arr, axis=2))
trajectories = np.concatenate(arrays, axis=0)
with open(f"data/merged/110-140.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [8]:
arrays = []
for i in ["110-130", "130-140"]:
    with open(f"data/merged/{i}.pickle", "rb") as f:
        arrays.append(pickle.load(f))
trajectories = np.concatenate(arrays, axis=2)
with open(f"data/merged/110-140.pickle", "wb") as f:
    pickle.dump(trajectories, f)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 12500 and the array at index 1 has size 11500